In [10]:
import os
import torch
from PIL import Image
from transformers import pipeline

input_dir = '/kaggle/input/datasets/yapeng111/enhanced-exdark-dataset/Enhanced_E_dataset' 
output_dir = '/kaggle/working/Depth_E'

os.makedirs(output_dir, exist_ok=True)
print(f"输出文件夹已准备: {output_dir}")

# ================= 加载模型 =================
print("⏳ 正在下载并加载 Depth Anything 模型 (首次运行需下载几百MB权重)...")
# 使用 task="depth-estimation" 直接调用管道，device=0 表示分配给 GPU
# 这里使用 small 版本，精度极高且防爆显存。如果显存充裕可以换成 base 版
depth_pipe = pipeline(task="depth-estimation", model="LiheYoung/depth-anything-small-hf", device=0)
print("✅ 模型加载成功！")

# ================= 搜索图片 =================
image_paths = []
# 遍历文件夹寻找图片（支持嵌套文件夹结构）
for root, _, files in os.walk(input_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(root, file))

print(f"🔎 共找到 {len(image_paths)} 张需要处理的图片。")

# ================= 批量推理 =================
if len(image_paths) > 0:
    print("🚀 开始生成深度图...")
    for i, img_path in enumerate(image_paths):
        # 保持文件命名与上一阶段完全一致，方便后续匹配成三元组 {L, D, T}
        file_name = os.path.basename(img_path)
        save_path = os.path.join(output_dir, file_name)
        
        try:
            # 1. 读取原图 (RGB格式)
            image = Image.open(img_path).convert('RGB')
            
            # --- 防爆显存机制 ---
            MAX_SIZE = 1200
            if max(image.size) > MAX_SIZE:
                resample_method = getattr(Image, 'Resampling', Image).LANCZOS
                image.thumbnail((MAX_SIZE, MAX_SIZE), resample_method)
            
            # 2. 模型推理获取深度图
            # pipeline 返回一个字典，里面的 "depth" 键对应一张 PIL 灰度图
            result = depth_pipe(image)
            depth_map = result["depth"] 
            
            # 3. 保存深度图 (ControlNet 通常需要单通道灰度图)
            depth_map.save(save_path)
            
        except Exception as e:
            print(f"❌ 处理图片 {file_name} 时出错: {e}")
            
        # 打印进度
        if (i + 1) % 100 == 0 or (i + 1) == len(image_paths):
            print(f"进度: 已处理 {i + 1} / {len(image_paths)} 张")

    print("🎉 增强图像的深度图 (D) 提取完毕！")
else:
    print("❌ 未找到图片，请检查 input_dir 路径是否正确。")

输出文件夹已准备: /kaggle/working/Depth_E
⏳ 正在下载并加载 Depth Anything 模型 (首次运行需下载几百MB权重)...


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

✅ 模型加载成功！
🔎 共找到 7363 张需要处理的图片。
🚀 开始生成深度图...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


进度: 已处理 100 / 7363 张
进度: 已处理 200 / 7363 张
进度: 已处理 300 / 7363 张
进度: 已处理 400 / 7363 张
进度: 已处理 500 / 7363 张
进度: 已处理 600 / 7363 张
进度: 已处理 700 / 7363 张
进度: 已处理 800 / 7363 张
进度: 已处理 900 / 7363 张
进度: 已处理 1000 / 7363 张
进度: 已处理 1100 / 7363 张
进度: 已处理 1200 / 7363 张
进度: 已处理 1300 / 7363 张
进度: 已处理 1400 / 7363 张
进度: 已处理 1500 / 7363 张
进度: 已处理 1600 / 7363 张
进度: 已处理 1700 / 7363 张
进度: 已处理 1800 / 7363 张
进度: 已处理 1900 / 7363 张
进度: 已处理 2000 / 7363 张
进度: 已处理 2100 / 7363 张
进度: 已处理 2200 / 7363 张
进度: 已处理 2300 / 7363 张
进度: 已处理 2400 / 7363 张
进度: 已处理 2500 / 7363 张
进度: 已处理 2600 / 7363 张
进度: 已处理 2700 / 7363 张
进度: 已处理 2800 / 7363 张
进度: 已处理 2900 / 7363 张
进度: 已处理 3000 / 7363 张
进度: 已处理 3100 / 7363 张
进度: 已处理 3200 / 7363 张
进度: 已处理 3300 / 7363 张
进度: 已处理 3400 / 7363 张
进度: 已处理 3500 / 7363 张
进度: 已处理 3600 / 7363 张
进度: 已处理 3700 / 7363 张
进度: 已处理 3800 / 7363 张
进度: 已处理 3900 / 7363 张
进度: 已处理 4000 / 7363 张
进度: 已处理 4100 / 7363 张
进度: 已处理 4200 / 7363 张
进度: 已处理 4300 / 7363 张
进度: 已处理 4400 / 7363 张
进度: 已处理 4500 / 7363 张
进度: 已处理 4600 / 7363

In [11]:
import shutil

print("正在打包深度图数据集...")
shutil.make_archive('/kaggle/working/Depth_E_dataset', 'zip', '/kaggle/working/Depth_E')
print("✅ 打包完成！请在右侧刷新并下载 Depth_E_dataset.zip")

正在打包深度图数据集...
✅ 打包完成！请在右侧刷新并下载 Depth_E_dataset.zip
